# V2 Training mit Kontext

Unterschiede zu V1 (`02_training.ipynb`):
1. Die Ausschnitte (256 × 256 px, 600 DPI) werden in jeder Epoche **neu aus den GeoTIFFs** geschnitten, keine PNGs:
   - jedes Label einmal pro Epoche, an einer zufälligen Stelle im Ausschnitt,
   - **Abdeckung**: ein Raster über die ganze Digitalisierungshülle jedes Plans, verteilt auf 15 Epochen. Nach 15 Epochen war jede Stelle jedes Plans genau einmal dabei (Text, Rahmen, Linien, Gebäude …).
2. **Ignore-Maske**: kein Loss im Randstreifen (halb abgeschnittene Symbole), ausserhalb der Hülle (dort wurde nicht geklickt) und um nicht digitalisierte PN-Punkte (dort steht oft ein echtes Symbol ohne Label).
3. **Kontext**: Die Kanäle 4–6 zeigen 1024 × 1024 px um dieselbe Mitte, auf 256 × 256 verkleinert (bei 1:1000 rund 43 m).
4. Die Hälfte der Ausschnitte wird als **300-DPI-Scan** simuliert.

Gleich wie V1, damit der Vergleich gültig bleibt: U-Net ResNet34 (ImageNet), Heatmap σ = 2 px, BCE mit `pos_weight` 100, Adam 1e-4, Batch 16.
Dataset und Hilfsfunktionen stehen in `grenzpunkte_v2.py` (DataLoader-Worker unter macOS).

Ergebnis: `unet_v2.pt` (bester Val-Loss), `unet_v2_letzte.pt` (letzte Epoche), `train_v2_verlauf.csv`. Dauer: etwa 12 min pro Epoche, rund 6 Stunden für 30 Epochen.
Auswerten mit `05_stufe2.ipynb` (`VERSION = "v2"`, mit und ohne `DPI = 300`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader

from grenzpunkte import BATCH_SIZE, device
from grenzpunkte_v2 import ZYKLUS, AusschnittDataset, AusschnittSampler, loss_fn

EPOCHS = 30  # = 2 Zyklen: jede Stelle jedes Trainingsplans kommt genau zweimal vor
LERNRATE = 1e-4

## Daten
Die Trainingsliste wird vor jeder Epoche neu gezogen. Die Validierung ist eine feste Auswahl: alle val-Labels und 3000 Abdeckungs-Ausschnitte, ohne Augmentierung.

Die DataLoader werden nur einmal erstellt, ihre Worker-Prozesse bleiben über alle Epochen bestehen (`persistent_workers=True`). Die aktuellen Ausschnitte bekommen sie über den `AusschnittSampler`. Mit neuen Workern in jeder Epoche blieben unter macOS jedes Mal Dateien offen, und das Training brach nach 13 Epochen mit «Too many open files» ab.

In [ ]:
train_dataset = AusschnittDataset("train", augment=True)
val_dataset = AusschnittDataset("val")
val_dataset.feste_auswahl()
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, sampler=AusschnittSampler(val_dataset),
                            num_workers=4, persistent_workers=True)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=AusschnittSampler(train_dataset, shuffle=True),
                              num_workers=4, persistent_workers=True)
print(f"Trainingspläne: {len(train_dataset.plaene)}, Validierung: {len(val_dataset)} Ausschnitte, "
      f"jede Stelle {EPOCHS / ZYKLUS:.1f}x im Training")

## Ein Beispiel ansehen
Ausschnitt mit Ziel-Heatmap (rote Kontur), Ignore-Maske (grün = Loss wird gerechnet) und Kontext (rot = Lage des Ausschnitts).

In [ ]:
train_dataset.epoche_vorbereiten(0)
X, y, maske = train_dataset[train_dataset.ausschnitte[0]]
X = X * torch.tensor([0.229, 0.224, 0.225] * 2)[:, None, None] + torch.tensor([0.485, 0.456, 0.406] * 2)[:, None, None]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(X[:3].permute(1, 2, 0).clamp(0, 1))
axes[0].contour(y[0], levels=[0.5], colors="red")
axes[1].imshow(X[:3].permute(1, 2, 0).clamp(0, 1))
axes[1].imshow(maske[0], alpha=0.4, cmap="RdYlGn", vmin=0, vmax=1)
axes[2].imshow(X[3:].permute(1, 2, 0).clamp(0, 1))
axes[2].add_patch(plt.Rectangle((95.5, 95.5), 64, 64, fill=False, ec="red"))
for ax, titel in zip(axes, ["Ausschnitt + Heatmap", "Ignore-Maske", "Kontext 1024 px"]):
    ax.set_title(titel)
    ax.axis("off")

## Modell und Optimizer
6 Eingangskanäle: `smp` passt die erste Schicht der ImageNet-Gewichte an (Gewichte wiederholt und skaliert). Der Loss (`loss_fn` in `grenzpunkte_v2.py`) ist die BCE von V1, gemittelt nur über die Pixel der Ignore-Maske.

In [ ]:
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=6, classes=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LERNRATE)

## Trainings- und Testschleife
Wie in V1, zusätzlich mit der Ignore-Maske.

In [ ]:
def train(dataloader, model, optimizer):
    size = len(dataloader.dataset)
    model.train()
    summe = 0
    for batch, (X, y, maske) in enumerate(dataloader):
        X, y, maske = X.to(device), y.to(device), maske.to(device)
        loss = loss_fn(model(X), y, maske)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        summe += loss.item()
        if batch % 100 == 0:
            print(f"loss: {loss.item():>7f}  [{(batch + 1) * len(X):>5d}/{size:>5d}]")
    return summe / len(dataloader)


def test(dataloader, model):
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y, maske in dataloader:
            X, y, maske = X.to(device), y.to(device), maske.to(device)
            test_loss += loss_fn(model(X), y, maske).item()
    test_loss /= len(dataloader)
    print(f"Val loss: {test_loss:>8f}\n")
    return test_loss

## Training

In [ ]:
verlauf = []
best = float("inf")
for t in range(EPOCHS):
    n_labels, n_raster = train_dataset.epoche_vorbereiten(t)
    print(f"Epoch {t+1}\n-------------------------------")
    print(f"Ausschnitte: {n_labels} um Labels, {n_raster} Abdeckung")
    train_loss = train(train_dataloader, model, optimizer)
    val_loss = test(val_dataloader, model)
    verlauf.append({"epoche": t + 1, "train_loss": train_loss, "val_loss": val_loss})
    pd.DataFrame(verlauf).to_csv("train_v2_verlauf.csv", index=False)
    if val_loss < best:
        best = val_loss
        torch.save(model.state_dict(), "unet_v2.pt")
    torch.save(model.state_dict(), "unet_v2_letzte.pt")
print("Done!")